# Chamfer count sensitivity to λ

For each method and sequence, the lower score is better.

$$S(\lambda)=G+\lambda B,\qquad B=\frac{\sum_{i,p}|n^{pred}_{i,p}-n^{ref}_{i,p}|}{N_{pred}+N_{ref}}.$$

$G$ is full bidirectional nearest-neighbor Chamfer distance in $(x,y,vt)$, averaged equally over shared polarities. The benchmark uses $v=1$ pixel/ms. $B$ compares ON and OFF event counts separately in each RGB frame interval $i$, normalized by the combined event count. At $\lambda=0$ you see geometry alone. A steeper line means a larger count penalty, and a crossing means two methods exchange ranks.

Use one **chamfer_count** results CSV. The plots reuse its saved components and require no GPU.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CSV_PATH = None  # Or set an explicit path to a chamfer_count results.csv
LAMBDAS = [0, 0.5, 1, 2, 5]
SAMPLE = None  # First available sequence, or a name such as "hand_1"

In [ ]:
if CSV_PATH is None:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "benchmarks" / "run_metric.py").is_file()), Path.cwd())
    candidates = sorted(root.glob("reports/hqevfi/chamfer_count/test-full-*/results.csv"))
    if len(candidates) != 1:
        raise ValueError("Set CSV_PATH to one chamfer_count CSV. Candidates: " + str(candidates))
    path = candidates[0]
else:
    path = Path(CSV_PATH).expanduser()

weights = np.unique(np.asarray(LAMBDAS, dtype=float))
if not weights.size or not np.isfinite(weights).all() or (weights < 0).any():
    raise ValueError("LAMBDAS must contain finite, nonnegative numbers.")
raw = pd.read_csv(path)
components = ["chamfer_geometry", "count_penalty"]
valid = raw.loc[raw["status"].eq("ok")].dropna(subset=components).copy()
if valid.empty or valid.duplicated(["sequence", "method"]).any():
    raise ValueError("Expected one valid row per sequence and method.")
scores = valid.set_index(["sequence", "method"])
scores = pd.DataFrame({w: scores.chamfer_geometry + w * scores.count_penalty for w in weights})
scores.columns.name = "λ"
sequences = sorted(valid.sequence.unique())
print(f"{path}\n{len(valid)} valid rows of {len(raw)}. Available sequences:")
print(", ".join(sequences))

In [ ]:
def show_scores(table, title):
    display(table.round(4))
    fig, ax = plt.subplots(figsize=(9, 4))
    for method, row in table.iterrows():
        ax.plot(weights, row, marker="o", label=method)
    ax.set(xlabel="Count weight λ", ylabel="Score (pixel-equivalent)", title=title)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    fig.tight_layout()
    plt.show()

# Compare means on the same sequences for every eligible method.
coverage = valid.groupby("sequence").method.nunique()
shared = coverage.index[coverage.eq(valid.method.nunique())]
if len(shared):
    mean_scores = scores.loc[scores.index.get_level_values("sequence").isin(shared)].groupby("method").mean()
    show_scores(mean_scores, f"Mean score across {len(shared)} shared sequences")
else:
    print("No sequences shared by all eligible methods. Individual sequences are available below.")

In [ ]:
sequence = sequences[0] if SAMPLE is None else SAMPLE
if sequence not in sequences:
    raise ValueError(f"Choose SAMPLE from {sequences}")
show_scores(scores.xs(sequence, level="sequence").sort_index(), f"Sequence: {sequence}")